In [ ]:
import os
import pandas as pd

# Set working directory to the course folder
os.chdir(
    r"Z:\File folders\Teaching\Reproducible Research\2023"
    r"\Repository\RRcourse2023\6. Coding and documentation"
)

# ---------------------------------------------------------------------------
# Load O*NET task data (ISCO-08 occupation level)
# Original data uses SOC classification, cross-walked to ISCO-08 via:
# https://ibs.org.pl/en/resources/occupation-classifications-crosswalks-from-onet-soc-to-isco/
# Columns: isco08 (occupation code), t_* (task intensity scores)
# ---------------------------------------------------------------------------
task_data = pd.read_csv(r"Data\onet_tasks.csv")

# ---------------------------------------------------------------------------
# Load Eurostat employment data (quarterly, 1-digit ISCO categories)
# Details: https://www.ilo.org/public/english/bureau/stat/isco/isco08/
# ---------------------------------------------------------------------------
ISCO_FILE = r"Data\Eurostat_employment_isco.xlsx"
ISCO_LEVELS = range(1, 10)  # ISCO categories 1-9

isco = {
    i: pd.read_excel(ISCO_FILE, sheet_name=f"ISCO{i}")
    for i in ISCO_LEVELS
}

# ---------------------------------------------------------------------------
# Calculate total employment per country across all ISCO categories.
# To add or remove countries, edit this list only — nothing else needs changing.
# ---------------------------------------------------------------------------
COUNTRIES = ["Belgium", "Spain", "Poland"]

totals = {
    country: sum(isco[i][country] for i in ISCO_LEVELS)
    for country in COUNTRIES
}

In [ ]:
# ---------------------------------------------------------------------------
# Add ISCO category labels and merge all occupation DataFrames into one
# ---------------------------------------------------------------------------
for i, df in isco.items():
    df["ISCO"] = i

all_data = pd.concat(isco.values(), ignore_index=True)

# ---------------------------------------------------------------------------
# Tile the per-country totals across all 9 ISCO rows and compute
# each occupation's share of total employment per country-period
# ---------------------------------------------------------------------------
for country in COUNTRIES:
    total_col = f"total_{country}"
    share_col = f"share_{country}"

    all_data[total_col] = pd.concat(
        [totals[country]] * len(ISCO_LEVELS), ignore_index=True
    )
    all_data[share_col] = all_data[country] / all_data[total_col]

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Task variables of interest: Non-Routine Cognitive Analytical (NRCA)
# Based on Autor & Acemoglu's task framework:
#   t_4A2a4  -- Analyzing Data or Information        (4.A.2.a.4)
#   t_4A2b2  -- Thinking Creatively                  (4.A.2.b.2)
#   t_4A4a1  -- Interpreting Meaning for Others      (4.A.4.a.1)
# To add more task categories, extend this list only.
# ---------------------------------------------------------------------------
TASK_VARS = ["t_4A2a4", "t_4A2b2", "t_4A4a1"]

# ---------------------------------------------------------------------------
# Extract 1-digit ISCO code and aggregate task means to that level
# ---------------------------------------------------------------------------
task_data["isco08_1dig"] = task_data["isco08"].astype(str).str[0].astype(int)

aggdata = (
    task_data
    .groupby("isco08_1dig")[TASK_VARS]   # select only the columns we need
    .mean()
)

# ---------------------------------------------------------------------------
# Merge occupation-level task scores into the employment panel
# ---------------------------------------------------------------------------
combined = pd.merge(
    all_data,
    aggdata,
    left_on="ISCO",
    right_on="isco08_1dig",
    how="left"
)

# ---------------------------------------------------------------------------
# Standardise each task variable separately per country
# (weighted mean = 0, weighted std = 1, weights = occupation employment share)
# Produces columns like std_Belgium_t_4A2a4, std_Poland_t_4A2b2, etc.
# ---------------------------------------------------------------------------
def weighted_standardise(series, weights):
    """Return a weighted z-score: (x - weighted_mean) / weighted_std."""
    mean = np.average(series, weights=weights)
    std  = np.sqrt(np.average((series - mean) ** 2, weights=weights))
    return (series - mean) / std

for country in COUNTRIES:
    for task in TASK_VARS:
        combined[f"std_{country}_{task}"] = weighted_standardise(
            combined[task],
            weights=combined[f"share_{country}"]
        )

In [ ]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Step 1: Compute NRCA score per country as sum of standardised task variables
# (Non-Routine Cognitive Analytical, per Autor & Acemoglu framework)
# ---------------------------------------------------------------------------
for country in COUNTRIES:
    std_cols = [f"std_{country}_{task}" for task in TASK_VARS]
    combined[f"{country}_NRCA"] = combined[std_cols].sum(axis=1)

# ---------------------------------------------------------------------------
# Step 2: Re-standardise the NRCA index (weighted, same approach as tasks)
# ---------------------------------------------------------------------------
for country in COUNTRIES:
    combined[f"std_{country}_NRCA"] = weighted_standardise(
        combined[f"{country}_NRCA"],
        weights=combined[f"share_{country}"]
    )

# ---------------------------------------------------------------------------
# Step 3: Weighted mean NRCA per time period
# Multiply standardised NRCA by occupation share, then sum within each period
# ---------------------------------------------------------------------------
agg_results = {}

for country in COUNTRIES:
    combined[f"multip_{country}_NRCA"] = (
        combined[f"std_{country}_NRCA"] * combined[f"share_{country}"]
    )
    agg_results[country] = (
        combined
        .groupby("TIME")[f"multip_{country}_NRCA"]
        .sum()
        .reset_index()
    )

# ---------------------------------------------------------------------------
# Step 4: Plot NRCA trend for each country
# Adding a country to COUNTRIES automatically adds a new panel here.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(len(COUNTRIES), 1, figsize=(10, 4 * len(COUNTRIES)), sharex=False)

for ax, country in zip(axes, COUNTRIES):
    agg = agg_results[country]
    col = f"multip_{country}_NRCA"
    ax.plot(agg["TIME"], agg[col])
    ax.set_xticks(range(0, len(agg), 3))
    ax.set_xticklabels(agg["TIME"].iloc[::3], rotation=45, ha="right")
    ax.set_title(f"Non-Routine Cognitive Analytical Task Intensity -- {country}")
    ax.set_ylabel("Weighted NRCA Index")
    ax.set_xlabel("Time")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------------
# To extend this analysis, add task vars above and/or countries to COUNTRIES.
# Example additional task group (Routine Manual):
#   t_4A3a3  -- Controlling Machines and Processes   (4.A.3.a.3)
#   t_4C2d1i -- Spend Time Making Repetitive Motions (4.C.2.d.1.i)
#   t_4C3d3  -- Pace Determined by Speed of Equipment(4.C.3.d.3)
# ---------------------------------------------------------------------------